In [11]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, top_k_accuracy_score,
    classification_report
)
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

In [12]:
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "rmisra/news-category-dataset",
    "News_Category_Dataset_v3.json",
    pandas_kwargs={"lines": True}
)

nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = [w for w in text.split() if w not in stop_words and len(w) > 2]
    return ' '.join(words)

df['text'] = (df['headline'].fillna('') + ' ' + df['short_description'].fillna(''))
df['text_clean'] = df['text'].apply(clean_text)
df = df[df['text_clean'].str.len() > 0].reset_index(drop=True)

Using Colab cache for faster access to the 'news-category-dataset' dataset.


In [13]:
counts = df['category'].value_counts()
keep = counts[counts >= 500].index
df = df[df['category'].isin(keep)].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    df['text_clean'], df['category'],
    test_size=0.2, random_state=42,
    stratify=df['category']
)

In [14]:
tfidf = TfidfVectorizer(
    max_features=100000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.7,
    sublinear_tf=True
)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

In [15]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, C=5.0, n_jobs=-1, random_state=42
    ),
    'Linear SVM': LinearSVC(
        C=1.0, max_iter=3000, random_state=42
    ),
    'Complement NB': ComplementNB(alpha=0.5),
    'LightGBM': lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.15,
    num_leaves=31,
    max_depth=8,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.3,
    reg_alpha=0.1,
    reg_lambda=0.1,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
    max_bin=63,
    force_row_wise=True,
    ),
}

results = []
for name, clf in models.items():
    print(f"\nОбучение {name}...")
    clf.fit(X_train_vec, y_train)
    y_pred = clf.predict(X_test_vec)

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    if hasattr(clf, 'predict_proba'):
        proba = clf.predict_proba(X_test_vec)
        top3 = top_k_accuracy_score(
            y_test, proba, k=3,
            labels=clf.classes_
        )
    else:
        top3 = np.nan

    results.append({
        'Модель': name,
        'Accuracy': acc,
        'F1 macro': f1_macro,
        'F1 weighted': f1_weighted,
        'Top-3 acc': top3
    })
    print(f"  Accuracy: {acc:.4f}  |  F1 macro: {f1_macro:.4f}  |  "
          f"F1 weighted: {f1_weighted:.4f}  |  Top-3: {top3:.4f}")



Обучение Logistic Regression...
  Accuracy: 0.6170  |  F1 macro: 0.4865  |  F1 weighted: 0.6003  |  Top-3: 0.8351

Обучение Linear SVM...
  Accuracy: 0.6147  |  F1 macro: 0.4849  |  F1 weighted: 0.5991  |  Top-3: nan

Обучение Complement NB...
  Accuracy: 0.5907  |  F1 macro: 0.4092  |  F1 weighted: 0.5465  |  Top-3: 0.8228

Обучение LightGBM...
  Accuracy: 0.5754  |  F1 macro: 0.4543  |  F1 weighted: 0.5599  |  Top-3: 0.7840


In [16]:
res_df = pd.DataFrame(results).sort_values('F1 macro', ascending=False)
print(res_df.round(4).to_string(index=False))

             Модель  Accuracy  F1 macro  F1 weighted  Top-3 acc
Logistic Regression    0.6170    0.4865       0.6003     0.8351
         Linear SVM    0.6147    0.4849       0.5991        NaN
           LightGBM    0.5754    0.4543       0.5599     0.7840
      Complement NB    0.5907    0.4092       0.5465     0.8228


In [17]:
best_name = res_df.iloc[0]['Модель']
best_model = models[best_name]
print(f"\nЛучшая модель: {best_name}")

y_pred_best = best_model.predict(X_test_vec)
top_categories = df['category'].value_counts().head(12).index.tolist()
print(f"\nClassification report (топ-12 категорий):")
print(classification_report(
    y_test, y_pred_best,
    labels=top_categories,
    zero_division=0
))


Лучшая модель: Logistic Regression

Classification report (топ-12 категорий):
                precision    recall  f1-score   support

      POLITICS       0.70      0.85      0.77      7120
      WELLNESS       0.57      0.78      0.66      3589
 ENTERTAINMENT       0.62      0.76      0.68      3472
        TRAVEL       0.70      0.80      0.75      1980
STYLE & BEAUTY       0.77      0.82      0.79      1963
     PARENTING       0.53      0.64      0.58      1758
HEALTHY LIVING       0.39      0.27      0.32      1338
  QUEER VOICES       0.79      0.66      0.72      1269
  FOOD & DRINK       0.63      0.71      0.67      1268
      BUSINESS       0.51      0.51      0.51      1198
        COMEDY       0.54      0.43      0.48      1079
        SPORTS       0.72      0.73      0.73      1015

     micro avg       0.64      0.73      0.68     27049
     macro avg       0.62      0.66      0.64     27049
  weighted avg       0.64      0.73      0.68     27049



In [18]:
def predict_top_k(text, k=3, model=None, vectorizer=None):
    """Top-k категорий с вероятностями для одного текста."""
    if model is None:
        model = best_model
    if vectorizer is None:
        vectorizer = tfidf

    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])

    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(vec)[0]
        top_idx = np.argsort(proba)[::-1][:k]
        return [(model.classes_[i], round(float(proba[i]), 4)) for i in top_idx]
    else:
        scores = model.decision_function(vec)[0]
        top_idx = np.argsort(scores)[::-1][:k]
        return [(model.classes_[i], round(float(scores[i]), 4)) for i in top_idx]

samples = [
    "Apple just announced the new iPhone with a faster chip and better camera.",
    "The president signed a new healthcare bill into law affecting millions.",
    "Scientists discovered a new species of deep-sea fish in the Pacific."
]

for s in samples:
    print(f"\nТекст: '{s}'")
    print("Топ-3 категории:")
    for cat, score in predict_top_k(s, k=3):
        print(f"  {cat:35s} — {score:.4f}")


Текст: 'Apple just announced the new iPhone with a faster chip and better camera.'
Топ-3 категории:
  TECH                                — 0.9646
  COMEDY                              — 0.0073
  WEIRD NEWS                          — 0.0040

Текст: 'The president signed a new healthcare bill into law affecting millions.'
Топ-3 категории:
  POLITICS                            — 0.7986
  WORLD NEWS                          — 0.0789
  HEALTHY LIVING                      — 0.0254

Текст: 'Scientists discovered a new species of deep-sea fish in the Pacific.'
Топ-3 категории:
  SCIENCE                             — 0.9368
  ENVIRONMENT                         — 0.0454
  GREEN                               — 0.0045
